# Kaggle full training pipeline (one notebook)

**Attach exactly two inputs** (Add Data, upload each zip as its own private dataset): `aspect_kaggle_bundle_v2.zip` and `dravidian_kaggle_bundle_v2.zip`.
**Settings:** Accelerator GPU (T4 or P100), Internet ON (XLM-R is downloaded from Hugging Face). Then Run All, or run the cells in order.

| Part | What it does | Expected outcome with the current data |
|---|---|---|
| 1 | GPU, dependencies, input discovery, file and CLI checks | must pass |
| 2 | Aspect readiness gate, then **aspect sentiment** XLM-R (top 2 layers), 459 agreed examples | trains; noisy metrics (below the size floors) |
| 3 | Aspect extraction: safe checks only | **no training** (mention counts far below the floors) |
| 4 | Aspect sequence / trajectory | **skipped** (50/20/20 series floors not met) |
| 5 | DravidianCodeMix XLM-R, Tamil, Malayalam, Kannada and joint, seeds 0 to 4, one folder per seed | trains; comment-level, NOT temporal |
| 6 | Collect checkpoints and results, summary report, `final_training_outputs.zip`, `.pt` checks | |

Not in this notebook: OET E4 retraining, Amazon + Dravidian joint multi-task training (needs a separate implementation). No threshold is lowered and `--allow_small_data` is never used. Nothing existing is overwritten: every experiment directory is new, and a re-run reuses finished ones.

## PART 1: Environment setup

In [ ]:
# 1.0 Configuration and helpers
import os, sys, json, shutil, subprocess, time, zipfile, traceback, platform, urllib.request
from pathlib import Path

DRY_RUN = os.environ.get("NB_DRY_RUN") == "1"      # local syntax/flow check only: commands are printed, nothing is trained
INPUT_ROOT = Path(os.environ.get("KAGGLE_INPUT_ROOT", "/kaggle/input"))
WORK = Path(os.environ.get("KAGGLE_WORK_ROOT", "/kaggle/working"))
PY = sys.executable
ASPECT_ZIP, DRAV_ZIP = "aspect_kaggle_bundle_v2.zip", "dravidian_kaggle_bundle_v2.zip"
ASPECT_REPO, DRAV_REPO = WORK / "aspect_repo", WORK / "dravidian_repo"      # separate trees: no conflicting source files
RAW, LOGS, HELPERS, FINAL = WORK / "raw_outputs", WORK / "stage_logs", WORK / "nb_helpers", WORK / "final_training_outputs"
for d in (RAW, LOGS, HELPERS):
    d.mkdir(parents=True, exist_ok=True)
ASPECT_OUT, DRAV_OUT, READY_OUT = RAW / "aspect_experiments", RAW / "dravidian_evaluation", RAW / "aspect_readiness"
SENT_NAME = "sentiment_xlmr_top2_agreed_only"
SEEDS = [0, 1, 2, 3, 4]
RUN_EXTRACTOR_PIPELINE_CHECK = False   # True only to smoke-test the extractor code path; its metrics would NOT be an evaluation
STATUS = {}                            # stage -> {"status", "detail"}; every stage reports here, nothing is swallowed


def record(stage, status, detail=""):
    STATUS[stage] = {"status": status, "detail": str(detail)[:2000]}
    print(f"[{stage}] {status}: {str(detail)[:400]}")


def run(cmd, cwd, log_file, env_extra=None):
    """Run a command, stream its output to the notebook and to a log file; return the exit code. Never trains in DRY_RUN."""
    print("$", " ".join(str(c) for c in cmd))
    if DRY_RUN:
        return 0
    env = {**os.environ, "PYTHONUNBUFFERED": "1", "TOKENIZERS_PARALLELISM": "false", **(env_extra or {})}
    Path(log_file).parent.mkdir(parents=True, exist_ok=True)
    with open(log_file, "a", encoding="utf-8") as lf:
        p = subprocess.Popen([str(c) for c in cmd], cwd=str(cwd), env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in p.stdout:
            sys.stdout.write(line)
            lf.write(line)
        return p.wait()


def tail(path, n=25):
    try:
        return "".join(Path(path).read_text(encoding="utf-8", errors="replace").splitlines(True)[-n:])
    except OSError:
        return "(no log)"


def check_cli(repo, script, args):
    """Every --flag in `args` must appear in the script's own --help output (so a wrong argument is caught before any training)."""
    r = subprocess.run([PY, script, "--help"], cwd=str(repo), capture_output=True, text=True)
    if r.returncode != 0:
        return [f"--help failed: {r.stderr[-300:]}"]
    import re
    return [a for a in args if str(a).startswith("--") and not re.search(r"(?<![\w-])" + re.escape(str(a)) + r"(?![\w-])", r.stdout)]


print("dry run:", DRY_RUN, "| python", platform.python_version(), "| work dir", WORK)

In [ ]:
# 1.1 GPU, dependencies (once), internet
import importlib

def gpu_check():
    try:
        import torch
    except ImportError as e:
        return False, f"torch is not importable: {e}"
    info = {"torch": torch.__version__, "cuda_available": torch.cuda.is_available()}
    if torch.cuda.is_available():
        info.update(device=torch.cuda.get_device_name(0), memory_gb=round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), cuda=torch.version.cuda)
    print(json.dumps(info, indent=1))
    try:
        print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout[:600])
    except OSError:
        pass
    return torch.cuda.is_available(), info

GPU_OK, GPU_INFO = gpu_check()
if not GPU_OK and not DRY_RUN:
    record("environment", "FAILED", "no CUDA GPU: set Accelerator to GPU T4/P100 in the notebook settings; all training in this notebook needs it")
    raise RuntimeError("No GPU. Training is never run on CPU.")

MARK = WORK / ".deps_installed"
if not MARK.exists() and not DRY_RUN:
    r = subprocess.run([PY, "-m", "pip", "install", "-q", "transformers", "scikit-learn", "pandas", "numpy"], capture_output=True, text=True)
    print(r.stdout[-500:], r.stderr[-500:])
    if r.returncode == 0:
        MARK.write_text("ok")
for m in ("torch", "transformers", "sklearn", "pandas", "numpy"):
    try:
        mod = importlib.import_module(m)
        print(m, getattr(mod, "__version__", "?"))
    except ImportError as e:
        record("environment", "FAILED", f"{m} missing: {e}")
        raise

INTERNET_OK = True
if not DRY_RUN:
    try:
        urllib.request.urlopen("https://huggingface.co", timeout=15)
    except Exception as e:
        INTERNET_OK = False
        record("environment", "FAILED", f"Hugging Face unreachable ({e}): turn Internet ON in the notebook settings; XLM-R must be downloaded")
        raise RuntimeError("Internet is off.")
record("environment", "SUCCESS", f"GPU {GPU_INFO if isinstance(GPU_INFO, str) else GPU_INFO.get('device', 'dry-run')}; internet {'ok' if INTERNET_OK else 'n/a'}")

In [ ]:
# 1.2 Locate the two attached bundles and extract them into SEPARATE directories
def find_bundle(zip_name, marker):
    zips, dirs = [], []
    for root, dnames, fnames in os.walk(INPUT_ROOT, followlinks=True):
        if zip_name in fnames:
            zips.append(Path(root) / zip_name)
        if (Path(root) / marker).exists():
            dirs.append(Path(root))
    return zips[0] if zips else (dirs[0] if dirs else None)


def prepare_repo(zip_name, marker, dest):
    if (dest / marker).exists():
        print(f"reusing {dest}")
        return dest
    src = find_bundle(zip_name, marker)
    if src is None:
        listing = [str(p) for p in INPUT_ROOT.rglob("*")][:40] if INPUT_ROOT.exists() else []
        raise FileNotFoundError(f"{zip_name} not found under {INPUT_ROOT}. Attach it as an input dataset. Found: {listing}")
    if dest.exists() and any(dest.iterdir()):
        raise FileExistsError(f"{dest} exists but is incomplete; remove it by hand (nothing is deleted automatically)")
    if src.is_file():
        with zipfile.ZipFile(src) as z:
            z.extractall(dest)
    else:
        shutil.copytree(src, dest)
    print(f"{zip_name}: {src} -> {dest}")
    return dest

try:
    prepare_repo(ASPECT_ZIP, "scripts/aspect_readiness.py", ASPECT_REPO)
    prepare_repo(DRAV_ZIP, "scripts/dravidian_evaluation.py", DRAV_REPO)
    INPUTS_OK = True
except Exception as e:
    INPUTS_OK = False
    record("inputs", "FAILED", e)
    traceback.print_exc()

In [ ]:
# 1.3 Verify every required script and data file, byte-compile the code (catches a Python-version or stale-bundle problem), check the CLI flags
REQ_ASPECT = ["scripts/train_aspect_sentiment.py", "scripts/train_aspect_extractor.py", "scripts/aspect_readiness.py", "scripts/validate_aspect_annotations.py",
              "src/aspect_data.py", "src/aspect_training.py", "src/aspect_sentiment_model.py", "src/aspect_extractor.py", "src/aspect_encoders.py",
              "src/experiment_utils.py", "src/eval_utils.py", "src/ontology.py", "ontology/product_aspect_taxonomy.json", "ontology/aspect_taxonomy.json",
              "data/annotations/pilot_batch_01_product_timelines.csv", "data/annotations/batch_02_wave1_reference.csv",
              "data/annotations/pilot_batch_01_annotator_A.csv", "data/annotations/pilot_batch_01_annotator_B.csv",
              "data/annotations/batch_02_wave1_annotator_1_DONE.csv", "data/annotations/batch_02_wave1_annotator_2_DONE.csv",
              "outputs/annotation_consolidation/pilot_batch_01/pilot_batch_01_PROVISIONAL_consolidated.csv",
              "outputs/annotation_consolidation/pilot_batch_01/pilot_batch_01_PROVISIONAL_provenance.csv"]
REQ_ASPECT += [f"data/annotations/training/aspect_training_{t}_{n}.csv" for t in ("sentiment", "extraction") for n in ("batch01", "wave1")]
REQ_DRAV = ["scripts/dravidian_evaluation.py", "scripts/eval_comment_level.py", "src/eval_utils.py"] + \
           [f"data/comment_level/{l}_{s}.csv" for l in ("tamil", "malayalam", "kannada") for s in ("train", "dev", "test")]
SENT_B01 = "data/annotations/training/aspect_training_sentiment_batch01.csv"
SENT_W1 = "data/annotations/training/aspect_training_sentiment_wave1.csv"
EXT_B01 = "data/annotations/training/aspect_training_extraction_batch01.csv"
EXT_W1 = "data/annotations/training/aspect_training_extraction_wave1.csv"

CMD_SENTIMENT = [PY, "-u", "scripts/train_aspect_sentiment.py", "--annotations", SENT_B01, SENT_W1, "--encoder", "xlmr", "--finetune", "partial",
                 "--finetune_layers", "2", "--class_weights", "sqrt", "--device", "cuda", "--epochs", "10", "--seed", "0", "--split_seed", "0",
                 "--output_root", str(ASPECT_OUT), "--name", SENT_NAME]
CMD_EXTRACTOR = [PY, "-u", "scripts/train_aspect_extractor.py", "--annotations", EXT_B01, EXT_W1, "--encoder", "xlmr", "--finetune", "partial",
                 "--finetune_layers", "2", "--device", "cuda", "--epochs", "10", "--seed", "0", "--split_seed", "0", "--output_root", str(ASPECT_OUT),
                 "--name", "extractor_xlmr_top2_PIPELINE_CHECK_ONLY"]


def cmd_dravidian(seed):
    return [PY, "-u", "scripts/dravidian_evaluation.py", "--encoder", "xlmr", "--seeds", str(seed), "--device", "cuda", "--out_dir", str(DRAV_OUT / f"seed_{seed}")]


CHECKS_OK = False
if INPUTS_OK:
    problems = []
    for repo, req in ((ASPECT_REPO, REQ_ASPECT), (DRAV_REPO, REQ_DRAV)):
        for rel in req:
            f = repo / rel
            if not f.exists() or f.stat().st_size == 0:
                problems.append(f"missing or empty: {repo.name}/{rel}")
        for py in list((repo / "scripts").glob("*.py")) + list((repo / "src").glob("*.py")):
            r = subprocess.run([PY, "-m", "py_compile", str(py)], capture_output=True, text=True)
            if r.returncode:
                problems.append(f"does not compile on Python {platform.python_version()}: {repo.name}/{py.name}: {r.stderr.strip()[-200:]}")
    if "save_head" not in (DRAV_REPO / "scripts/dravidian_evaluation.py").read_text(encoding="utf-8"):
        problems.append("dravidian bundle is the old version without checkpoint saving: attach dravidian_kaggle_bundle_v2.zip")
    for repo, script, cmd in ((ASPECT_REPO, "scripts/train_aspect_sentiment.py", CMD_SENTIMENT), (ASPECT_REPO, "scripts/train_aspect_extractor.py", CMD_EXTRACTOR),
                              (DRAV_REPO, "scripts/dravidian_evaluation.py", cmd_dravidian(0))):
        bad = check_cli(repo, script, cmd[3:])
        if bad:
            problems.append(f"{script}: arguments not accepted by the script: {bad}")
    if problems:
        record("checks", "FAILED", "; ".join(problems))
        print("\n".join(problems))
    else:
        CHECKS_OK = True
        record("checks", "SUCCESS", "all required files exist, all code compiles, every CLI flag is accepted")

## PART 2: Amazon aspect sentiment (XLM-R, top 2 layers, agreed-only examples)

In [ ]:
# 2.1 Readiness gate (model-free). Stops the aspect stages on structural errors or leakage; the size warnings are reported, not hidden.
READY = None
GATE_OK = False
HELPER_READY = '''
import json, sys
from collections import Counter
repo, out = sys.argv[1], sys.argv[2]
sys.path[:0] = [repo + "/scripts", repo + "/src"]
import aspect_readiness as ar
import aspect_data as ad
import aspect_training as at
gate = ar.main([])
s, e = at.load_corpus(ar.SENT), at.load_corpus(ar.EXT)
products = sorted({m.entity_id for m in s.messages})
ext_splits = at.make_splits(e.messages, (0.7, 0.15, 0.15), 0)
mentions = {n: sum(len(m.mentions) for m in ms if m.status == "ASPECT_PRESENT") for n, ms in ext_splits.items()}
esum = at.split_summary(ext_splits)
ext = {"mentions": mentions, "products_total": len({m.entity_id for m in e.messages}), "products_per_split": {n: esum[n]["n_products"] for n in esum},
       "floors": {"total_products": at.MIN_TOTAL_PRODUCTS, "eval_products": at.MIN_EVAL_PRODUCTS, "eval_mentions": at.MIN_EVAL_ITEMS, "train_mentions": at.MIN_TRAIN_ITEMS}}
ext["floors_met"] = (ext["products_total"] >= at.MIN_TOTAL_PRODUCTS and min(ext["products_per_split"].get(n, 0) for n in ("val", "test")) >= at.MIN_EVAL_PRODUCTS
                     and min(mentions["val"], mentions["test"]) >= at.MIN_EVAL_ITEMS and mentions["train"] >= at.MIN_TRAIN_ITEMS)
seq = {}
for seed in range(6):
    outer = ad.split_entities(products, (0.85, 0.15), seed, ("pool", "test"))
    inner = ad.split_entities(outer["pool"], (0.8, 0.2), seed, ("train", "val"))
    counts = {}
    for n, ps in {"train": inner["train"], "val": inner["val"], "test": outer["test"]}.items():
        es = ad.build_sentiment_examples([m for m in s.messages if m.entity_id in set(ps)], level="message_category")
        ser = Counter((x.entity_id, x.category) for x in es if x.label in ("POSITIVE", "NEGATIVE", "MIXED"))
        counts[n] = sum(v >= 2 for v in ser.values())
    seq[str(seed)] = {"usable_series": counts, "blockers": at.series_blockers(counts)}
n_ex = len(ad.build_sentiment_examples(s.messages, level="message_category"))
json.dump({"gate": gate, "extraction": ext, "sequence": seq, "oof_blockers": at.cv_blockers(len(products), n_ex, 5), "sentiment_examples": n_ex,
           "products": len(products)}, open(out, "w"), indent=1)
'''
if CHECKS_OK:
    READY_OUT.mkdir(parents=True, exist_ok=True)
    (HELPERS / "readiness_json.py").write_text(HELPER_READY, encoding="utf-8")
    ready_json = READY_OUT / "readiness.json"
    if ready_json.exists():
        ready_json = READY_OUT / f"readiness_{int(time.time())}.json"
    rc = run([PY, str(HELPERS / "readiness_json.py"), str(ASPECT_REPO), str(ready_json)], ASPECT_REPO, READY_OUT / "readiness.txt")
    if DRY_RUN:
        record("aspect_readiness", "DRY_RUN", "readiness helper not executed in the dry run's run() wrapper; running it directly (it never trains)")
        rc = subprocess.run([PY, str(HELPERS / "readiness_json.py"), str(ASPECT_REPO), str(ready_json)], cwd=str(ASPECT_REPO), capture_output=True, text=True)
        print(rc.stdout[-3000:], rc.stderr[-500:])
        rc = rc.returncode
    if rc == 0 and ready_json.exists():
        READY = json.load(open(ready_json))
        GATE_OK = all(READY["gate"].values())
        if GATE_OK:
            record("aspect_readiness", "SUCCESS", f"data, extraction safety and leakage checks pass; {READY['sentiment_examples']} sentiment examples, {READY['products']} products")
        else:
            record("aspect_readiness", "FAILED", f"structural or leakage problem, aspect stages will not run: {READY['gate']}")
    else:
        record("aspect_readiness", "FAILED", tail(READY_OUT / "readiness.txt"))
else:
    record("aspect_readiness", "SKIPPED", "environment or file checks failed")

In [ ]:
# 2.2 Train aspect sentiment (existing script, existing settings; best-validation checkpoint). Never overwrites an experiment directory.
SENT_DIR = ASPECT_OUT / SENT_NAME
SENT_EXPECT = ["config.json", "splits.json", "class_weights.json", "train_log.jsonl", "history.json", "best_checkpoint.pt", "best_validation.json",
               "test_metrics.json", "summary.json"]
SENT_SUMMARY = None
stage = "aspect_sentiment"
if not (CHECKS_OK and GATE_OK):
    record(stage, "SKIPPED", "readiness gate or environment checks did not pass")
elif DRY_RUN:
    run(CMD_SENTIMENT, ASPECT_REPO, LOGS / f"{stage}.log")
    record(stage, "DRY_RUN", "command printed, not executed")
elif SENT_DIR.exists():
    ok = (SENT_DIR / "summary.json").exists() and all((SENT_DIR / f).exists() for f in SENT_EXPECT)
    record(stage, "SUCCESS" if ok else "INCOMPLETE", f"{SENT_DIR} already exists and is never overwritten ({'finished run reused' if ok else 'unfinished, inspect it'})")
    SENT_SUMMARY = json.load(open(SENT_DIR / "summary.json")) if ok else None
else:
    ASPECT_OUT.mkdir(parents=True, exist_ok=True)
    try:
        rc = run(CMD_SENTIMENT, ASPECT_REPO, LOGS / f"{stage}.log")
        missing = [f for f in SENT_EXPECT if not (SENT_DIR / f).exists() or (SENT_DIR / f).stat().st_size == 0]
        if rc != 0:
            record(stage, "FAILED", f"exit code {rc}; missing outputs {missing}\n{tail(LOGS / f'{stage}.log')}")
        elif missing:
            record(stage, "INCOMPLETE", f"finished but missing {missing}")
        else:
            SENT_SUMMARY = json.load(open(SENT_DIR / "summary.json"))
            record(stage, "SUCCESS", f"best epoch {SENT_SUMMARY['best_epoch']}; validation {SENT_SUMMARY['validation']}; test {SENT_SUMMARY['test']}; "
                                     f"too small for meaningful evaluation: {SENT_SUMMARY['dataset_too_small_for_meaningful_evaluation']}")
    except Exception as e:
        record(stage, "FAILED", f"{e}\n{traceback.format_exc()[-800:]}")
if SENT_SUMMARY and SENT_SUMMARY.get("warnings"):
    print("Size warnings (the metrics below are a pipeline result, not strong evidence):")
    for w in SENT_SUMMARY["warnings"]:
        print("  -", w)

In [ ]:
# 2.3 Slim checkpoint: the trained top-2 layers plus the head only (the frozen XLM-R weights are the public model and are not duplicated).
# The full best_checkpoint.pt stays untouched in its experiment directory; the slim file is checked to rebuild the full state exactly.
SLIM_PATH = SENT_DIR / "best_checkpoint_slim.pt"
stage = "aspect_sentiment_slim_checkpoint"
if STATUS.get("aspect_sentiment", {}).get("status") != "SUCCESS" or DRY_RUN:
    record(stage, "SKIPPED" if not DRY_RUN else "DRY_RUN", "needs a finished aspect_sentiment run")
elif SLIM_PATH.exists():
    record(stage, "SUCCESS", "already exists, not overwritten")
else:
    try:
        import torch
        from transformers import AutoModel
        full = torch.load(SENT_DIR / "best_checkpoint.pt", map_location="cpu", weights_only=False)
        state = full["model_state_dict"]
        if not full.get("includes_encoder"):
            record(stage, "SKIPPED", "the checkpoint holds no encoder weights, it is already slim")
        else:
            base = {"encoder." + k: v for k, v in AutoModel.from_pretrained("xlm-roberta-base").state_dict().items()}
            kept = {k: v for k, v in state.items() if not k.startswith("encoder.") or k not in base or not torch.equal(v, base[k])}
            rebuilt = {**{k: v for k, v in base.items() if k in state}, **kept}
            same = set(rebuilt) == set(state) and all(torch.equal(rebuilt[k], state[k]) for k in state)
            if not same:
                record(stage, "FAILED", "slim checkpoint could not reproduce the full state; the full best_checkpoint.pt is the deliverable")
            else:
                slim = {k: v for k, v in full.items() if k != "model_state_dict"}
                slim.update(model_state_dict=kept, slim=True, encoder_base="xlm-roberta-base", full_checkpoint="best_checkpoint.pt",
                            kept_tensors=len(kept), dropped_identical_encoder_tensors=len(state) - len(kept))
                torch.save(slim, SLIM_PATH)
                record(stage, "SUCCESS", f"{SLIM_PATH.stat().st_size / 1e6:.0f} MB vs {(SENT_DIR / 'best_checkpoint.pt').stat().st_size / 1e6:.0f} MB full; "
                                         f"kept {len(kept)} tensors, dropped {len(state) - len(kept)} identical pretrained tensors; rebuild verified")
    except Exception as e:
        record(stage, "FAILED", f"{e}\n{traceback.format_exc()[-800:]}")

## PART 3: Amazon aspect extraction (safe checks only)

In [ ]:
# 3.1 The extraction files hold only messages both annotators finished with IDENTICAL spans, so no disputed or unfinished text becomes background (O) labels
# (checked in the readiness gate). Floors are compared with the existing constants; training is NOT started. Optional smoke test: RUN_EXTRACTOR_PIPELINE_CHECK.
HELPER_BIO = '''
import json, sys
repo, out = sys.argv[1], sys.argv[2]
sys.path[:0] = [repo + "/scripts", repo + "/src"]
import aspect_readiness as ar, aspect_data as ad, aspect_training as at
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained("xlm-roberta-base")
assert getattr(tok, "is_fast", False)
c = at.load_corpus(ar.EXT)
ex, rep = ad.build_extraction_examples(c.messages, tok, max_length=128)
tags = ad.bio_tagset()
n_b = sum(1 for e in ex for l in e.encoding.labels if l >= 0 and tags[l].startswith("B-"))
json.dump({"examples": len(ex), "report": rep, "B_tags": n_b,
           "alignment_ok": rep.get("unaligned_mentions", 0) == 0 and rep.get("truncated_mentions", 0) == 0}, open(out, "w"), indent=1)
'''
stage = "aspect_extraction"
if not (CHECKS_OK and GATE_OK and READY):
    record(stage, "SKIPPED", "readiness gate or environment checks did not pass")
elif DRY_RUN:
    record(stage, "DRY_RUN", f"floors met: {READY['extraction']['floors_met']}; mentions per split {READY['extraction']['mentions']}; BIO check needs the tokenizer download, not run here")
else:
    ext = READY["extraction"]
    try:
        (HELPERS / "bio_check.py").write_text(HELPER_BIO, encoding="utf-8")
        bio_json = READY_OUT / "extraction_bio_check.json"
        if not bio_json.exists():
            rc = run([PY, str(HELPERS / "bio_check.py"), str(ASPECT_REPO), str(bio_json)], ASPECT_REPO, LOGS / "aspect_extraction_bio_check.log")
            if rc != 0:
                raise RuntimeError(tail(LOGS / "aspect_extraction_bio_check.log"))
        bio = json.load(open(bio_json))
        msg = (f"safe checks done: {bio['examples']} messages BIO-encoded with the XLM-R tokenizer ({bio['B_tags']} B tags, alignment ok: {bio['alignment_ok']}). "
               f"Mentions per split {ext['mentions']} against floors (eval {ext['floors']['eval_mentions']}, train {ext['floors']['train_mentions']}); "
               f"products {ext['products_total']} (floor {ext['floors']['total_products']}). Floors met: {ext['floors_met']}.")
        if ext["floors_met"] and not RUN_EXTRACTOR_PIPELINE_CHECK:
            msg += " Floors are met, so set RUN_EXTRACTOR_PIPELINE_CHECK or train it deliberately; not started automatically."
        record(stage, "SKIPPED" if not RUN_EXTRACTOR_PIPELINE_CHECK else "RUNNING", "no extraction evaluation is claimed. " + msg)
        if RUN_EXTRACTOR_PIPELINE_CHECK:
            d = ASPECT_OUT / "extractor_xlmr_top2_PIPELINE_CHECK_ONLY"
            if d.exists():
                record(stage, "INCOMPLETE" if not (d / "summary.json").exists() else "SUCCESS", f"{d} already exists, not overwritten; PIPELINE CHECK ONLY, metrics are not an evaluation")
            else:
                rc = run(CMD_EXTRACTOR, ASPECT_REPO, LOGS / f"{stage}.log")
                record(stage, "SUCCESS" if rc == 0 and (d / "summary.json").exists() else "FAILED",
                       "PIPELINE CHECK ONLY (validation and test mentions are far below the floors, metrics are not an evaluation)" + ("" if rc == 0 else "\n" + tail(LOGS / f"{stage}.log")))
    except Exception as e:
        record(stage, "FAILED", f"{e}\n{traceback.format_exc()[-800:]}")

## PART 4: Aspect sequence / trajectory (readiness only)

In [ ]:
stage = "aspect_sequence"
if not READY:
    record(stage, "SKIPPED", "readiness could not be computed")
else:
    seq = READY["sequence"]
    for seed, v in seq.items():
        print(f"split_seed {seed}: usable series {v['usable_series']}  {v['blockers'] or 'ok'}")
    passing = [s for s, v in seq.items() if not v["blockers"]]
    if passing:
        record(stage, "SKIPPED", f"floors are met on split_seed {passing}, but the sequence model needs out-of-fold predictions and a separate run that this notebook deliberately does not include")
    else:
        record(stage, "SKIPPED", "50/20/20 usable-series floors are NOT met on any tested product split; no thresholds were lowered and --allow_small_data is not used. "
                                 f"OOF generation blockers: {READY['oof_blockers'] or 'none'} (OOF is not run because its only consumer, the sequence model, is blocked)")

## PART 5: DravidianCodeMix XLM-R (comment-level sentiment; no timestamps or sequences exist and none are invented)

In [ ]:
# 5.1 One run per seed in its own directory, so a completed seed is preserved even if a later one fails. Each run trains the Tamil, Malayalam, Kannada
# and joint heads and saves a checkpoint for each, plus metrics, predictions, configuration and confusion matrices.
DRAV_EXPECT = ["config/config.json", "config/data_report.json", "summary.md"] + [f"metrics/{n}.json" for n in
               ("tamil_to_tamil", "malayalam_to_malayalam", "kannada_to_kannada", "joint_to_tamil", "joint_to_malayalam", "joint_to_kannada")] + \
              [f"predictions/{n}_test.csv" for n in ("tamil_to_tamil", "malayalam_to_malayalam", "kannada_to_kannada", "joint_to_tamil", "joint_to_malayalam", "joint_to_kannada")]
DRAV_PT = lambda seed: [f"checkpoints/{s}_seed{seed}.pt" for s in ("tamil", "malayalam", "kannada", "joint")]
DRAV_DONE = []
for seed in SEEDS:
    stage = f"dravidian_seed_{seed}"
    d = DRAV_OUT / f"seed_{seed}"
    if not CHECKS_OK:
        record(stage, "SKIPPED", "environment or file checks failed")
        continue
    if DRY_RUN:
        run(cmd_dravidian(seed), DRAV_REPO, LOGS / f"{stage}.log")
        record(stage, "DRY_RUN", "command printed, not executed")
        continue
    try:
        if d.exists():
            missing = [f for f in DRAV_EXPECT + DRAV_PT(seed) if not (d / f).exists() or (d / f).stat().st_size == 0]
            record(stage, "SUCCESS" if not missing else "INCOMPLETE", f"{d} already exists and is never overwritten; " + ("finished seed reused" if not missing else f"missing {missing}"))
        else:
            DRAV_OUT.mkdir(parents=True, exist_ok=True)
            rc = run(cmd_dravidian(seed), DRAV_REPO, LOGS / f"{stage}.log")
            missing = [f for f in DRAV_EXPECT + DRAV_PT(seed) if not (d / f).exists() or (d / f).stat().st_size == 0]
            if rc != 0:
                record(stage, "FAILED", f"exit code {rc}; missing {missing}\n{tail(LOGS / f'{stage}.log')}")
            elif missing:
                record(stage, "INCOMPLETE", f"missing {missing}")
            else:
                record(stage, "SUCCESS", "4 setups trained, checkpoints and results saved")
    except Exception as e:
        record(stage, "FAILED", f"{e}\n{traceback.format_exc()[-800:]}")
    if STATUS[stage]["status"] == "SUCCESS":
        DRAV_DONE.append(seed)

In [ ]:
# 5.2 Aggregate the finished seeds (mean and standard deviation over seeds; confusion matrices summed over seeds). Only real saved metrics are used.
import numpy as np
AGG = {}
SETUPS = ["tamil_to_tamil", "malayalam_to_malayalam", "kannada_to_kannada", "joint_to_tamil", "joint_to_malayalam", "joint_to_kannada"]
for name in SETUPS:
    rows = []
    for seed in DRAV_DONE:
        f = DRAV_OUT / f"seed_{seed}" / "metrics" / f"{name}.json"
        if f.exists():
            rows.append(json.load(open(f)))
    if not rows:
        continue
    g = lambda fn: [fn(r) for r in rows]
    cm = np.sum([np.asarray(r["per_seed_reports"][next(iter(r["per_seed_reports"]))]["four_class"]["confusion_matrix"]) for r in rows], axis=0)
    AGG[name] = {"seeds": [s for s in DRAV_DONE], "n_seeds": len(rows), "test_rows": rows[0]["test_rows"],
                 "accuracy_4class_mean": float(np.mean(g(lambda r: r["four_class"]["accuracy_mean"]))), "accuracy_4class_std": float(np.std(g(lambda r: r["four_class"]["accuracy_mean"]))),
                 "macro_f1_4class_mean": float(np.mean(g(lambda r: r["four_class"]["macro_f1_mean"]))), "macro_f1_4class_std": float(np.std(g(lambda r: r["four_class"]["macro_f1_mean"]))),
                 "macro_f1_3class_mean": float(np.mean(g(lambda r: r["primary_3class"]["macro_f1_mean"]))), "macro_f1_3class_std": float(np.std(g(lambda r: r["primary_3class"]["macro_f1_mean"]))),
                 "per_class_f1_mean": {c: float(np.mean(g(lambda r: r["four_class"]["per_class_f1_mean"][c]))) for c in rows[0]["four_class"]["per_class_f1_mean"]},
                 "majority_macro_f1_4class": rows[0]["majority_baseline_4class"]["macro_f1"], "confusion_matrix_4class_summed_over_seeds": cm.tolist()}
    print(f"{name:24s} seeds {len(rows)} | acc {AGG[name]['accuracy_4class_mean']:.3f} | 4-class macro-F1 {AGG[name]['macro_f1_4class_mean']:.3f} +- {AGG[name]['macro_f1_4class_std']:.3f} "
          f"| 3-class macro-F1 {AGG[name]['macro_f1_3class_mean']:.3f} | majority {AGG[name]['majority_macro_f1_4class']:.3f}")
if not AGG:
    print("no finished Dravidian seed to aggregate")

## PART 6: Collect results, summary report, zip, checkpoint checks

In [ ]:
# 6.1 Collect into /kaggle/working/final_training_outputs (an earlier folder is moved aside, never deleted or overwritten), write the report, build the zip, check every .pt
try:
    if FINAL.exists() and any(FINAL.iterdir()):
        FINAL.rename(WORK / f"final_training_outputs_prev_{int(time.time())}")
    FINAL.mkdir(parents=True, exist_ok=True)
    slim_ok = SLIM_PATH.exists() and STATUS.get("aspect_sentiment_slim_checkpoint", {}).get("status") == "SUCCESS"

    expected_pt = []                                   # (relative path inside FINAL, description)
    if SENT_DIR.exists():
        shutil.copytree(SENT_DIR, FINAL / "aspect_sentiment" / SENT_NAME, ignore=shutil.ignore_patterns("best_checkpoint.pt") if slim_ok else None)
    expected_pt.append((f"aspect_sentiment/{SENT_NAME}/" + ("best_checkpoint_slim.pt" if slim_ok else "best_checkpoint.pt"), "aspect sentiment best-validation checkpoint"))
    for sub in ("aspect_extraction", "aspect_sequence"):
        (FINAL / sub).mkdir(exist_ok=True)
    if READY_OUT.exists():
        shutil.copytree(READY_OUT, FINAL / "aspect_readiness")
    for seed in SEEDS:
        d = DRAV_OUT / f"seed_{seed}"
        if d.exists():
            shutil.copytree(d, FINAL / "dravidian" / f"seed_{seed}")
        expected_pt += [(f"dravidian/seed_{seed}/{p}", f"Dravidian head, seed {seed}") for p in DRAV_PT(seed)]
    if AGG:
        (FINAL / "dravidian").mkdir(exist_ok=True)
        json.dump(AGG, open(FINAL / "dravidian" / "aggregate_over_seeds.json", "w"), indent=1)
    if LOGS.exists():
        shutil.copytree(LOGS, FINAL / "stage_logs")

    pt_check = []
    for rel, what in expected_pt:
        f = FINAL / rel
        pt_check.append({"file": rel, "what": what, "exists": f.exists(), "bytes": f.stat().st_size if f.exists() else 0})
    pt_ok = all(c["exists"] and c["bytes"] > 0 for c in pt_check)

    groups = {"successful": [], "failed": [], "skipped": [], "incomplete": [], "dry_run": []}
    for k, v in STATUS.items():
        groups[{"SUCCESS": "successful", "FAILED": "failed", "SKIPPED": "skipped", "INCOMPLETE": "incomplete", "DRY_RUN": "dry_run", "RUNNING": "incomplete"}.get(v["status"], "incomplete")].append(k)
    report = {"generated": time.strftime("%Y-%m-%d %H:%M:%S"), "dry_run": DRY_RUN, "gpu": GPU_INFO if isinstance(GPU_INFO, dict) else str(GPU_INFO), "stages": STATUS, "groups": groups,
              "aspect_sentiment": ({"best_epoch": SENT_SUMMARY["best_epoch"], "validation": SENT_SUMMARY["validation"], "test": SENT_SUMMARY["test"],
                                    "warnings": SENT_SUMMARY["warnings"]} if SENT_SUMMARY else None),
              "dravidian_aggregate": AGG, "checkpoint_files": pt_check, "all_expected_checkpoints_present_and_non_empty": pt_ok,
              "not_run": ["OET E4 retraining", "aspect sequence / trajectory (floors not met)", "OOF generation", "Amazon + Dravidian joint multi-task training (design only)"]}
    json.dump(report, open(FINAL / "summary_report.json", "w"), indent=1)
    L = ["# Training run summary", "", f"Generated {report['generated']}{' (DRY RUN: nothing was trained)' if DRY_RUN else ''}.", "", "## Stages"]
    for g, names in groups.items():
        L.append(f"* **{g}** ({len(names)}): " + (", ".join(names) if names else "none"))
    L += ["", "| stage | status | detail |", "|---|---|---|"] + [f"| {k} | {v['status']} | {v['detail'].splitlines()[0][:300] if v['detail'] else ''} |" for k, v in STATUS.items()]
    if SENT_SUMMARY:
        L += ["", "## Aspect sentiment (agreed-only 459 examples; validation 74 and test 67 examples, below the 100 floor: noisy, a pipeline result)",
              f"best epoch {SENT_SUMMARY['best_epoch']}; validation {SENT_SUMMARY['validation']}; test {SENT_SUMMARY['test']}"]
    if AGG:
        L += ["", "## DravidianCodeMix XLM-R (comment-level; mean +- std over finished seeds)", "| setup -> test | seeds | acc (4-class) | macro-F1 (4-class) | macro-F1 (3-class) | majority macro-F1 |", "|---|---|---|---|---|---|"]
        L += [f"| {k} | {v['n_seeds']} | {v['accuracy_4class_mean']:.3f} | {v['macro_f1_4class_mean']:.3f} +- {v['macro_f1_4class_std']:.3f} | {v['macro_f1_3class_mean']:.3f} +- {v['macro_f1_3class_std']:.3f} | {v['majority_macro_f1_4class']:.3f} |" for k, v in AGG.items()]
        L += ["", "Per-seed metrics, predictions and confusion matrices are in `dravidian/seed_<k>/`; summed confusion matrices are in `dravidian/aggregate_over_seeds.json`. This is a comment-level experiment; it does not show temporal opinion evolution."]
    L += ["", f"## Checkpoints: {'all expected files present and non-empty' if pt_ok else 'PROBLEM, see below'}"] + [f"* {'OK ' if c['exists'] and c['bytes'] else 'MISSING '}{c['file']} ({c['bytes'] / 1e6:.2f} MB)" for c in pt_check]
    L += ["", "## Not run", *[f"* {x}" for x in report["not_run"]]]
    (FINAL / "summary_report.md").write_text("\n".join(L) + "\n", encoding="utf-8")

    zip_path = WORK / "final_training_outputs.zip"
    if zip_path.exists():
        zip_path.rename(WORK / f"final_training_outputs_prev_{int(time.time())}.zip")
    shutil.make_archive(str(WORK / "final_training_outputs"), "zip", str(FINAL))
    with zipfile.ZipFile(zip_path) as z:
        bad = z.testzip()
        names = set(z.namelist())
        in_zip = [c["file"] for c in pt_check if c["file"] in names]
        has_full = any(n.endswith("best_checkpoint.pt") for n in names)
    zip_ok = bad is None and len(in_zip) == len([c for c in pt_check if c["exists"]]) and not (slim_ok and has_full)
    record("final_collection", "SUCCESS" if (pt_ok and zip_ok) else "INCOMPLETE",
           f"zip {zip_path} ({zip_path.stat().st_size / 1e6:.0f} MB), {len(in_zip)}/{len(pt_check)} expected .pt files inside; zip test {'ok' if bad is None else bad}; "
           f"pretrained-weight duplicate excluded: {slim_ok}")
    print("\n".join(L))
except Exception as e:
    record("final_collection", "FAILED", f"{e}\n{traceback.format_exc()[-1200:]}")
    traceback.print_exc()